# Node 01 · Finalize — 冻结 Node 01
### 化学身份 → 空间结构身份的过渡

本 notebook 调度 `core/Node01_finalize.py`。它【不做化学判断】——
所有电荷/质子化/立体计数已由 `Node01_microstate_enumerate.py` v0.8.0 冻结。
Finalize 只做下游整形：排除 C 类、立体枚举、ID 方案、下游标记。

**输入**：v0.8.0 `master_microstates.csv`（1606 分子，0 冲突，QC 通过）
**输出**：`V3_microstate_table_final.csv`（~2030 行）+ `node01_excluded.csv`（14）+ manifest

**策略（三方商定）**：
- 排除 `review_tier=='C_exclude'`（14 肼鎓）
- 立体：k=0 单行；1≤k≤3 自动枚举 `_s01/_s02/...`（补零，parent_id 回指）；k≥4 保留+标记+不进 docking（方案B）
- 极端电荷 |q|≥3 → `extreme_charge_flag`（不排除，供 pre-docking 审视）

In [1]:
# ── Cell 1 · 配置 ──────────────────────────────────────
from pathlib import Path
import sys
DRY_RUN = False   # 先预览；确认后改 False 写盘
PROJECT_ROOT = Path("/mnt/hpc/home/wanghongmei/lang2").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# v0.8.0 指认输出（Node 01 identification 的结果）
INPUT_CSV = PROJECT_ROOT / "results" / "01_microstates" / "master_microstates.csv"
OUT_DIR   = PROJECT_ROOT / "results" / "01_microstates"
print("DRY_RUN   =", DRY_RUN)
print("INPUT_CSV =", INPUT_CSV, " 存在:", INPUT_CSV.is_file())
print("OUT_DIR   =", OUT_DIR)

DRY_RUN   = False
INPUT_CSV = /mnt/hpc/home/wanghongmei/lang2/results/01_microstates/master_microstates.csv  存在: True
OUT_DIR   = /mnt/hpc/home/wanghongmei/lang2/results/01_microstates


In [2]:
# ── Cell 2 · 导入 finalize 模块 + 读输入 ────────────────
import importlib
from core import Node01_finalize as F
importlib.reload(F)
import pandas as pd
from rdkit import Chem
print("Node01_finalize 版本:", F.FINALIZE_VERSION)
print("  STEREO_AUTO_MAX (k≤此值自动枚举):", F.STEREO_AUTO_MAX)
print("  EXTREME_CHARGE_ABS (|q|≥此值标记):", F.EXTREME_CHARGE_ABS)
df = pd.read_csv(INPUT_CSV, dtype=str)
print(f"\n读入 {len(df)} 行；review_tier 分布:")
print(df["review_tier"].value_counts().to_string())

Node01_finalize 版本: 0.2.1
  STEREO_AUTO_MAX (k≤此值自动枚举): 3
  EXTREME_CHARGE_ABS (|q|≥此值标记): 3

读入 1606 行；review_tier 分布:
review_tier
A_resolved    1489
B_manual       117


In [3]:
# ── Cell 3 · 运行 finalize（纯内存，不写盘）─────────────
final_df, excluded_df, stats = F.finalize(df, Chem, pd)
print("=== finalize 统计 ===")
for k, v in stats.items():
    if k != "stereo_enumeration_failures":
        print(f"  {k}: {v}")
if stats["stereo_enumeration_failures"]:
    print(f"  枚举失败(退回方案B): {stats['stereo_enumeration_failures']}")
print(f"\n最终表 stereo_status 分布:")
print(final_df["stereo_status"].value_counts().to_string())

=== finalize 统计 ===
  n_input: 1606
  n_excluded_classC: 0
  n_kept_parents: 1606
  n_stereo_auto_parents: 95
  n_stereo_isomers_generated: 279
  n_stereo_held_k_ge_4: 4
  n_no_stereo: 1504
  n_extreme_charge: 19
  n_overprotonation_flagged: 1
  n_stereo_collapsed_single: 3
  n_stereo_truncated: 0
  stereo_truncated_parents: []
  n_final_rows: 1790
  downstream_eligibility_counts: {'READY': 1785, 'HOLD_STEREO': 4, 'HOLD_CHARGE_REVIEW': 1}
  charge_risk_counts: {'normal': 1759, 'high': 31}

最终表 stereo_status 分布:
stereo_status
defined_or_none                    1504
auto_enumerated                     279
held_high_stereo                      4
no_real_stereo_after_correction       3


In [ ]:
# ── Cell 4 · 完整性检查 ────────────────────────────────
# v0.2.2 修复(checkagain33):排除数改为对照 finalize 统计,不再硬编码"应 14"。
# "14"是更早计划池的陈旧数字;当前 1606 scope 下肼鎓数为 0。
# 一个与现实静默相悖的硬编码期望,正是日后会掩盖真实回归的东西。
exp_c = stats["n_excluded_classC"]
print("完整性检查:")
print(f"  name 唯一:        {final_df['name'].is_unique}  (应 True)")
print(f"  parent_id 全填充: {final_df['parent_id'].notna().all()}  (应 True)")
print(f"  排除表行数:       {len(excluded_df)}  (finalize 声明 {exp_c})")
assert len(excluded_df) == exp_c, "excluded 行数与 finalize 统计不符"
print(f"  最终行数:         {len(final_df)}  (finalize 声明 {stats['n_final_rows']})")
assert len(final_df) == stats["n_final_rows"], "最终行数与 finalize 统计不符"
print(f"  needs_stereo_review=True: {final_df['needs_stereo_review'].sum()} (k≥4 保留)")
ec = final_df[final_df['extreme_charge_flag']==True]
print(f"  extreme_charge 行: {len(ec)}  (来自 {ec['parent_id'].nunique()} 个母体)")
# 抽查 ID 方案
print("\n抽查枚举 ID 方案:")
enum = final_df[final_df['stereo_status']=='auto_enumerated']
if len(enum):
    p = enum['parent_id'].iloc[0]
    kids = enum[enum['parent_id']==p]
    print(f"  {p} → {list(kids['name'])}")

In [5]:
# ── Cell 5 · 写盘（仅 DRY_RUN=False）────────────────────
manifest, paths = F.write_outputs(final_df, excluded_df, stats,
                                  OUT_DIR, INPUT_CSV, dry_run=DRY_RUN)
if not DRY_RUN:
    print("✓ 已写:")
    for k, p in paths.items():
        print(f"  {k}: {p}")
    print("\n✅ Node 01 已冻结。下一步：Node 02 — 3D 权威分子构造。")
else:
    print("[DRY_RUN] 未写盘。manifest 预览:\n")
    import json
    print(json.dumps(manifest, indent=2, ensure_ascii=False)[:1500])
    print("\n→ 确认无误后，把 Cell 1 的 DRY_RUN 改 False，重跑 Cell 3-5 写盘。")

✓ 已写:
  final: /mnt/hpc/home/wanghongmei/lang2/results/01_microstates/V3_microstate_table_final.csv
  excluded: /mnt/hpc/home/wanghongmei/lang2/results/01_microstates/node01_excluded.csv
  manifest: /mnt/hpc/home/wanghongmei/lang2/results/01_microstates/Node01_finalize_manifest.json

✅ Node 01 已冻结。下一步：Node 02 — 3D 权威分子构造。
